# 🌫️ Módulo 05: Modelos Generativos y Espacios Latentes
## Capítulo 2: Fundamentos de Modelos de Difusión (DDPM): Termodinámica, Forward Process y Denoising Inverso

> *"Destruir información es trivial: basta con añadir una pizca de ruido gaussiano en cada paso hasta que una fotografía se convierta en pura estática cósmica. La genialidad de Jascha Sohl-Dickstein y Jonathan Ho consistió en invertir la flecha del tiempo de la termodinámica: si entrenas una red neuronal para predecir exactamente el ruido añadido en cada instante, puedes partir de estática pura y esculpir, paso a paso, una imagen fotorrealista. Bienvenidos a la física de la difusión."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/05_generative_models/02_diffusion_foundations_ddpm.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos las librerías matemáticas y fijamos semillas para asegurar reproducibilidad determinista.

In [ ]:
# !pip install -q numpy matplotlib torch
from typing import Tuple, List, Dict, Optional
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para estudiar Modelos de Difusión (DDPM) from scratch")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### La Inspiración en la Termodinámica (Sohl-Dickstein et al., 2015)
En 2015, **Jascha Sohl-Dickstein** (físico y neurocientífico de Stanford) publicó un artículo seminal titulado *"Deep Unsupervised Learning using Nonequilibrium Thermodynamics"*:
* Observó la difusión molecular en física: si viertes una gota de tinta en agua, las moléculas se dispersan gradualmente hasta alcanzar un estado de equilibrio de máxima entropía (ruido térmico uniforme).
* El proceso hacia adelante (*Forward*) destruye la estructura paso a paso según leyes estadísticas bien conocidas.
* **La gran hipótesis:** Si podemos invertir la flecha del tiempo aprendiendo la dinámica inversa (*Reverse Process*), podemos partir del caos térmico y reconstituir la estructura original.

### La Revolución de DDPM: Jonathan Ho et al. (NeurIPS 2020)
Durante 5 años, el trabajo de Sohl-Dickstein pasó casi desapercibido por su complejidad teórica, mientras las GANs dominaban la generación de imágenes. En 2020, **Jonathan Ho, Ajay Jain y Pieter Abbeel** de UC Berkeley publicaron **DDPM (Denoising Diffusion Probabilistic Models)**:
1. Demostraron que la complicada cota variacional se simplifica radicalmente: **basta con entrenar una red para predecir el ruido residual $\epsilon$ mediante un simple Error Cuadrático Medio (MSE)**.
2. Superaron la calidad visual de las GANs **sin entrenamiento adversarial**, eliminando por completo el colapso de modos (*mode collapse*) y logrando un entrenamiento estable y convergente.
3. Este descubrimiento cimentó el nacimiento de **Stable Diffusion, Midjourney, DALL-E 3 y Sora**.

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### El Proceso Hacia Adelante (*Forward Process* $q$)
Dada una muestra de datos real $x_0$, añadimos gradualmente ruido gaussiano a lo largo de $T$ pasos temporales ($t = 1, \dots, T$):
$$q(x_t | x_{t-1}) = \mathcal{N}\left(x_t; \sqrt{1 - \beta_t} x_{t-1}, \beta_t I\right)$$
Donde $\beta_1, \beta_2, \dots, \beta_T$ es un programa de varianzas (*variance schedule*).

### El Truco Matemático del Salto Temporal Directo ($O(1)$)
¿Tenemos que simular paso a paso $x_1, x_2, \dots, x_t$ para llegar al paso $t$? ¡No!
Definiendo $\alpha_t = 1 - \beta_t$ y el producto acumulado $\bar{lpha}_t = \prod_{s=1}^t \alpha_s$:
Gracias a la propiedad aditiva de las variables gaussianas independientes, podemos muestrear directamente $x_t$ a partir de $x_0$ para cualquier instante $t$ en un solo paso:
$$\mathbf{x_t = \sqrt{\bar{lpha}_t} x_0 + \sqrt{1 - \bar{lpha}_t} \epsilon, \quad \text{donde } \epsilon \sim \mathcal{N}(0, I)}$$

* Cuando $t = 0$: $\bar{lpha}_0 \approx 1 \implies x_t = x_0$ (datos puros).
* Cuando $t = T$: $\bar{lpha}_T \approx 0 \implies x_T \approx \epsilon$ (ruido blanco puro $\mathcal{N}(0, I)$).

### El Proceso Inverso (*Denoising Reverse Process* $p_\theta$)
Entrenamos una red neuronal $\epsilon_\theta(x_t, t)$ cuyo único cometido es **adivinar qué ruido $\epsilon$ fue inyectado** para transformar $x_0$ en $x_t$:
$$\mathbf{\mathcal{L}_{simple} = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]}$$

### El Algoritmo Generativo de Muestreo:
Para generar una nueva muestra desde cero:
1. Muestreamos estática pura $x_T \sim \mathcal{N}(0, I)$.
2. Para $t = T, T-1, \dots, 1$, restamos una fracción del ruido predicho por la red:
   $$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{lpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z, \quad z \sim \mathcal{N}(0, I)$$
3. Al llegar a $t=0$, obtenemos una muestra generada cristalina y fotorrealista.

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

Construyamos la clase `DDPMFromScratch` y una red predictora de ruido con embeddings temporales en PyTorch puro.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """
    Codificación posicional sinusoidal para inyectar el paso temporal t en la red.
    """
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half_dim = self.dim // 2
        emb = np.log(10000.0) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, dtype=torch.float32, device=t.device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)


class DenoiseMLP(nn.Module):
    """
    Red neuronal predictora de ruido epsilon_theta(x_t, t).
    """
    def __init__(self, data_dim: int = 2, time_dim: int = 16, hidden_dim: int = 64):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbedding(time_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim)
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_embed = self.time_emb(t)
        # Concatenar vector ruidoso y vector de tiempo
        inp = torch.cat([x_t, t_embed], dim=-1)
        return self.net(inp)


class DDPMFromScratch:
    """
    Implementación completa del algoritmo DDPM (Ho et al., 2020).
    """
    def __init__(self, timesteps: int = 100, beta_start: float = 1e-4, beta_end: float = 0.02):
        self.timesteps = timesteps
        # 1. Schedule lineal de betas
        self.betas = torch.linspace(beta_start, beta_end, timesteps)
        # 2. Alphas y productos acumulados
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        
        # Coeficientes para el muestreo directo q(x_t | x_0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

    def q_sample(self, x_0: torch.Tensor, t: torch.Tensor, noise: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward Process directo O(1): x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
        """
        if noise is None:
            noise = torch.randn_like(x_0)
            
        sqrt_alpha = self.sqrt_alphas_cumprod[t].unsqueeze(-1)
        sqrt_one_minus_alpha = self.sqrt_one_minus_alphas_cumprod[t].unsqueeze(-1)
        x_t = sqrt_alpha * x_0 + sqrt_one_minus_alpha * noise
        return x_t, noise

    @torch.no_grad()
    def p_sample_loop(self, model: nn.Module, shape: Tuple[int, int]) -> List[torch.Tensor]:
        """
        Reverse Process: parte de x_T ~ N(0, I) y genera x_0 paso a paso.
        """
        model.eval()
        x = torch.randn(shape)
        intermediates = [x.clone()]
        
        for t in reversed(range(self.timesteps)):
            t_tensor = torch.full((shape[0],), t, dtype=torch.long)
            pred_noise = model(x, t_tensor)
            
            beta_t = self.betas[t]
            alpha_t = self.alphas[t]
            sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alphas_cumprod[t]
            
            # Media del paso inverso
            mean = (1.0 / torch.sqrt(alpha_t)) * (x - (beta_t / sqrt_one_minus_alpha_bar_t) * pred_noise)
            
            if t > 0:
                z = torch.randn_like(x)
                sigma_t = torch.sqrt(beta_t)
                x = mean + sigma_t * z
            else:
                x = mean
                
            if t % (self.timesteps // 5) == 0 or t == 0:
                intermediates.append(x.clone())
                
        return intermediates

print("✅ DenoiseMLP y DDPMFromScratch compilados exitosamente")

### Entrenamiento en un Manifold 2D (Dos Espirales Concéntricas)
Entrenemos el modelo para aprender a reconstruir una figura geométrica bidimensional no trivial a partir de ruido puro:

In [ ]:
# Generar datos de prueba: círculo exterior y círculo interior
np.random.seed(42)
n_pts = 600
theta1 = np.random.uniform(0, 2*np.pi, n_pts // 2)
r1 = np.random.normal(2.0, 0.1, n_pts // 2)
theta2 = np.random.uniform(0, 2*np.pi, n_pts // 2)
r2 = np.random.normal(0.8, 0.1, n_pts // 2)

x1, y1 = r1 * np.cos(theta1), r1 * np.sin(theta1)
x2, y2 = r2 * np.cos(theta2), r2 * np.sin(theta2)
data_2d = np.vstack([np.column_stack([x1, y1]), np.column_stack([x2, y2])]).astype(np.float32)
t_data = torch.from_numpy(data_2d)

# Inicializar DDPM y red de desruidificación
ddpm = DDPMFromScratch(timesteps=80)
model_diff = DenoiseMLP(data_dim=2, time_dim=16, hidden_dim=128)
opt_diff = torch.optim.Adam(model_diff.parameters(), lr=2e-3)

print("Entrenando modelo de difusión en 300 épocas...")
model_diff.train()
for epoch in range(300):
    opt_diff.zero_grad()
    # Muestrear t aleatorio uniforme para cada muestra del lote
    t_batch = torch.randint(0, ddpm.timesteps, (len(t_data),), dtype=torch.long)
    x_noisy, target_noise = ddpm.q_sample(t_data, t_batch)
    
    # Predecir ruido
    pred_noise = model_diff(x_noisy, t_batch)
    loss = F.mse_loss(pred_noise, target_noise)
    loss.backward()
    opt_diff.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Época {epoch+1}/300 | Pérdida MSE de Ruido: {loss.item():.4f}")

print("✅ Modelo de Difusión entrenado exitosamente")

### Experimento Visual: La Inversión del Tiempo Termodinámico
Generamos una nube de puntos partiendo de ruido gaussiano blanco puro ($t = 80$) y visualizamos cómo se van disipando las partículas de ruido hasta reconstituir la figura original ($t = 0$):

In [ ]:
samples_history = ddpm.p_sample_loop(model_diff, shape=(500, 2))

n_snaps = len(samples_history)
fig, axes = plt.subplots(1, n_snaps, figsize=(3.2 * n_snaps, 3.2))

for i, snap in enumerate(samples_history):
    pts = snap.numpy()
    axes[i].scatter(pts[:, 0], pts[:, 1], c='crimson', s=6, alpha=0.7)
    axes[i].set_xlim(-3.5, 3.5)
    axes[i].set_ylim(-3.5, 3.5)
    if i == 0:
        axes[i].set_title("Paso T=80\n(Ruido Blanco Puro)", fontsize=10, fontweight='bold')
    elif i == n_snaps - 1:
        axes[i].set_title("Paso t=0\n(Muestra Generada)", fontsize=10, fontweight='bold')
    else:
        axes[i].set_title(f"Paso intermedio #{i}", fontsize=10)
    axes[i].grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()
print("🚀 ¡La estática aleatoria se autoorganiza paso a paso en los anillos concéntricos aprendidos!")

---

## 4. ⚡ Transición a PyTorch Moderno

Verifiquemos las propiedades estadísticas del Forward Process de difusión:

In [ ]:
# Verificar que para t=T, la varianza de x_T es exactamente 1.0 (N(0, I))
x_zeros = torch.zeros(5000, 2)
t_final = torch.full((5000,), ddpm.timesteps - 1, dtype=torch.long)
x_noisy_final, _ = ddpm.q_sample(x_zeros, t_final)

std_final = x_noisy_final.std().item()
mean_final = x_noisy_final.mean().item()
print(f"Media en t=T: {mean_final:.4f} (Esperada: 0.0)")
print(f"Desviación estándar en t=T: {std_final:.4f} (Esperada: 1.0)")
assert np.isclose(mean_final, 0.0, atol=0.1)
assert np.isclose(std_final, 1.0, atol=0.1)
print("✅ La difusión hacia adelante converge rigurosamente a una distribución gaussiana estándar N(0, I)")

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Comparativa de Schedules: Lineal vs Coseno
En 2021, Alex Nichol y Prafulla Dhariwal (*"Improved Denoising Diffusion Probabilistic Models"*) descubrieron que el schedule lineal destruye la señal demasiado rápido al inicio.
Propusieron el **Schedule Coseno**:
$$\bar{\alpha}_t = \frac{f(t)}{f(0)}, \quad f(t) = \cos\left( \frac{t/T + s}{1 + s} \cdot \frac{\pi}{2} \right)^2$$

Grafiquemos ambos schedules para comparar la retención de señal:

In [ ]:
T_sched = 100
steps = np.arange(T_sched)

# 1. Schedule Lineal
betas_lin = np.linspace(1e-4, 0.02, T_sched)
alpha_bars_lin = np.cumprod(1.0 - betas_lin)

# 2. Schedule Coseno
s = 0.008
f_t = np.cos(((steps / T_sched) + s) / (1 + s) * np.pi / 2) ** 2
alpha_bars_cos = f_t / f_t[0]

plt.figure(figsize=(8, 4))
plt.plot(steps, alpha_bars_lin, 'r--', label='Schedule Lineal (Ho et al. 2020) - Rápida destrucción inicial')
plt.plot(steps, alpha_bars_cos, 'g-', linewidth=2, label='Schedule Coseno (Nichol & Dhariwal 2021) - Caída suave y progresiva')
plt.title('Conservación de Señal en Forward Process: alpha_bar_t', fontsize=11, fontweight='bold')
plt.xlabel('Paso Temporal (t)')
plt.ylabel('Proporción de Señal Original (alpha_bar_t)')
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend()
plt.show()

### Reto 2 (Para resolver): Implementar Classifier-Free Guidance (CFG)
En 2022, Jonathan Ho y Tim Salimans revolucionaron la generación condicionada (como Text-to-Image en Stable Diffusion) con **CFG**.
En cada paso, el modelo calcula dos predicciones de ruido: una incondicional (con prompt vacío $\emptyset$) y otra condicionada con el prompt $c$:
$$\mathbf{\tilde{\epsilon} = \epsilon_\theta(x_t, \emptyset) + w \cdot \left( \epsilon_\theta(x_t, c) - \epsilon_\theta(x_t, \emptyset) \right)}$$
Donde $w > 1.0$ (típicamente $w \in [5, 8]$) empuja la generación hacia el concepto $c$ aumentando drásticamente la nitidez y adherencia al prompt.

Implementa a continuación la función `apply_cfg(noise_uncond, noise_cond, guidance_scale=7.5)`:

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def apply_cfg(noise_uncond: torch.Tensor, noise_cond: torch.Tensor, guidance_scale: float = 7.5) -> torch.Tensor:
    """
    Aplica Classifier-Free Guidance combinando las predicciones de ruido.
    """
    # Tu implementación aquí
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Sohl-Dickstein, J., et al. (2015):** *"Deep Unsupervised Learning using Nonequilibrium Thermodynamics"*, ICML 2015. [arXiv:1503.03585](https://arxiv.org/abs/1503.03585)
   * *¿Qué leer?* La conexión física entre difusión molecular, reversión temporal y modelos probabilísticos.
2. **Ho, J., Jain, A., & Abbeel, P. (2020):** *"Denoising Diffusion Probabilistic Models"* (DDPM), NeurIPS 2020. [arXiv:2006.11239](https://arxiv.org/abs/2006.11239)
   * *¿Qué leer?* Algoritmos 1 (Training) y 2 (Sampling): las dos páginas que cambiaron la historia de la IA generativa.
3. **Nichol, A. Q., & Dhariwal, P. (2021):** *"Improved Denoising Diffusion Probabilistic Models"*, ICML 2021. [arXiv:2102.09672](https://arxiv.org/abs/2102.09672)
   * *¿Qué leer?* La derivación del schedule coseno de $\bar{\alpha}_t$ y la parametrización de varianzas aprendidas.
4. **Ho, J., & Salimans, T. (2022):** *"Classifier-Free Diffusion Guidance"*, NeurIPS Workshop. [arXiv:2207.12598](https://arxiv.org/abs/2207.12598)
   * *¿Qué leer?* El truco universal que permite a Stable Diffusion y Midjourney obedecer al texto del usuario.